# 🎓 Yonsei Colab Studio — 교육용 소형 LLM 파인튜닝

Google Colab **무료 T4 GPU**에서 [Unsloth](https://github.com/unslothai/unsloth)로 소형 LLM을 파인튜닝합니다.
메모리 사용량 ~80% 절감, 속도 2~3배로 무료 코랩에서도 OOM 없이 학습됩니다.

- **목적**: 20가지 교육 시나리오(소크라테스 문답 · 단계별 채점 · 학부모 안내문 변환 등) 특화
- **기본 모델**: `unsloth/Qwen2.5-1.5B-Instruct` (한국어 우수, 초경량)
- **방식**: LoRA(QLoRA) 4bit

> ⚠️ 먼저 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택하세요.


## 0. GPU 확인


In [ ]:
!nvidia-smi


## 1. Unsloth 설치
코랩 최신 환경에 맞춰 자동 설치됩니다. (1~2분 소요)


In [ ]:
%%capture
import os
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets


## 2. 학습 데이터 — HuggingFace Hub에서 받기
손으로 만든 예시가 아니라 **실제 한국어 교육 데이터셋**을 HF Hub에서 받아 20시나리오 포맷으로 변환합니다.

추천 데이터셋(소스별 N개만 streaming 추출 → 대형도 전체 다운로드 없음):
- `JosephLee/korean-socratic-qa` (105k) — 소크라테스 문답
- `kuotient/orca-math-word-problems-193k-korean` (193k) — 수학 단계 풀이
- `beomi/KoAlpaca-v1.1a` (21k) — 일반 instruction 베이스
- `jojo0217/korean_safe_conversation` (27k, Apache-2.0) — 정서지원·공감
- `neuralfoundry-coder/aihub-korean-education-instruct-sample` (6k) — 교육 상담·분석


In [ ]:
# 저장소 clone (스크립트/레지스트리 사용)
REPO = 'https://github.com/xide-projext/xideproject.git'
import os
if not os.path.exists('xideproject'):
    !git clone -q $REPO
%cd xideproject

# HF에서 소스별 8000개씩 받아 data/hf_train.jsonl 생성 (≈5분)
!python scripts/fetch_hf_datasets.py --per-source 8000 --val-ratio 0.05

from pathlib import Path
DATA_PATH = "data/hf_train.jsonl"
assert Path(DATA_PATH).exists(), "fetch 실패 — 위 셀 로그 확인"
print("학습 데이터 준비 완료:", DATA_PATH)


## 3. 모델 로드 (4bit)


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"  # 더 작게: Qwen2.5-0.5B-Instruct / 다른 계열: Llama-3.2-1B-Instruct

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # 자동 감지 (T4=float16)
    load_in_4bit = True,
)


## 4. LoRA 어댑터 추가
전체 가중치가 아닌 일부 행렬만 학습해 메모리/시간을 절약합니다.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


## 5. 데이터 포맷팅 (채팅 템플릿)
`instruction`(시나리오 지시) → system, `input` → user, `output` → assistant 로 매핑합니다.


In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(ex):
    msgs = [
        {"role": "system",    "content": ex["instruction"]},
        {"role": "user",      "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.map(to_text)
print("샘플 수:", len(dataset))
print(dataset[0]["text"][:600])


## 6. 트레이너 설정 & 학습
데모용 `max_steps=60`. 실제 학습에선 `num_train_epochs`로 바꾸세요.


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                # 또는 num_train_epochs = 3 (데이터 많을 때)
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


## 7. 추론 테스트
학습한 시나리오 톤이 나오는지 즉석에서 확인합니다.


In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "시나리오[1]: 소크라테스식 문답법으로 학생을 가이드하세요. 정답을 바로 주지 말고 힌트와 역질문으로 스스로 답을 찾게 유도합니다."},
    {"role": "user",   "content": "선생님, 삼각형 내각의 합이 왜 180도예요?"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))


## 8. 저장 & 내보내기
- **LoRA 어댑터**: 가볍게 재사용 (수십 MB)
- **GGUF**: Ollama / LM Studio / llama.cpp 용 (옵션, 시간 소요)


In [ ]:
# (1) LoRA 어댑터만 저장
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("LoRA 저장 완료 → lora_model/")

# (2) GGUF 내보내기 (필요할 때 주석 해제)
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

# (3) 구글 드라이브에 백업 (필요할 때 주석 해제)
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r lora_model /content/drive/MyDrive/


---
### 다음 단계
1. 데이터를 더 받으려면 `--per-source` 를 키우세요 (예: 20000). 소스 추가는 `scripts/fetch_hf_datasets.py` 의 SOURCES 레지스트리에서.
2. 특정 시나리오만 학습하려면 `--only socratic math` 처럼 지정.
3. 시나리오별 성능은 `data/scenarios.json` 의 *기대치(expectation)* 기준으로 평가하세요.
4. `data/seed_train.jsonl` 은 학습용이 아니라 시나리오별 **목표 톤 예시(참고용)** 입니다.
